In [1]:
import sys
import os
from langchain.chat_models import init_chat_model

from dotenv import load_dotenv

load_dotenv(override=True)

True

# Util

In [2]:
import os
from pathlib import Path
from typing import Optional
from typing import List

def get_file_path(file_path: str) -> str:
    """
    파일 경로를 절대 경로로 변환하는 함수
    
    Args:
        file_path: 상대 경로 또는 절대 경로
    """
    # 파일 경로 확인 및 절대 경로로 변환
    file_path = Path(file_path)
    if not file_path.is_absolute():
        # 노트북 위치 기준 상대 경로 처리
        # 노트북은 asset_ai_portal/tests 폴더에 있고, documents는 20_code_test 루트에 있음
        current_dir = Path.cwd()
        
        # asset_ai_portal/tests에서 실행 중이면 상위로 두 번 이동 (20_code_test 루트)
        if current_dir.name == 'tests' and current_dir.parent.name == 'asset_ai_portal':
            project_root = current_dir.parent.parent  # tests -> asset_ai_portal -> 20_code_test
        elif current_dir.name == 'asset_ai_portal':
            project_root = current_dir.parent  # asset_ai_portal -> 20_code_test
        else:
            # 20_code_test에서 실행 중이면 그대로 사용
            project_root = current_dir
        
        file_path = project_root / file_path
    
    if not file_path.exists():
        raise FileNotFoundError(f"PDF 파일을 찾을 수 없습니다: {file_path}")
    
    return str(file_path)


def table_to_markdown(table: List[List]) -> str:
    """
    표 데이터를 마크다운 테이블 형식으로 변환하는 헬퍼 함수
    
    Args:
        table: 2차원 리스트 형태의 표 데이터
    
    Returns:
        마크다운 테이블 문자열
    """
    if not table or len(table) == 0:
        return ""
    
    # 빈 셀을 빈 문자열로 변환
    def clean_cell(cell):
        if cell is None:
            return ""
        return str(cell).strip()
    
    # 표 데이터 정리
    cleaned_table = [[clean_cell(cell) for cell in row] for row in table]
    
    # 최대 컬럼 수 확인
    max_cols = max(len(row) for row in cleaned_table) if cleaned_table else 0
    
    # 모든 행을 동일한 컬럼 수로 맞춤
    normalized_table = []
    for row in cleaned_table:
        normalized_row = row + [""] * (max_cols - len(row))
        normalized_table.append(normalized_row)
    
    if not normalized_table:
        return ""
    
    markdown_lines = []
    
    # 헤더 행 (첫 번째 행)
    header = normalized_table[0]
    markdown_lines.append("| " + " | ".join(header) + " |")
    
    # 구분선
    markdown_lines.append("| " + " | ".join(["---"] * len(header)) + " |")
    
    # 데이터 행들
    for row in normalized_table[1:]:
        markdown_lines.append("| " + " | ".join(row) + " |")
    
    return "\n".join(markdown_lines)

# LLM 모델 생성

In [3]:
# 문서가 변액일임펀드 설정/해지 지시서인지 확인하는 LLM 노드 생성

from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
# LLM 모델 정의

LLM_MODEL = os.getenv("LLM_MODEL")
LLM_BASE_URL=os.getenv("LLM_BASE_URL")
LLM_API_KEY=os.getenv("LLM_API_KEY")
LLM_TEMPERATURE=os.getenv("LLM_TEMPERATURE")

def create_llm_model():
    # vLLM 모델 인스턴스 생성
    llm = init_chat_model(
        "openai:",
        temperature=LLM_TEMPERATURE,
        top_p=0.1,  # top_p는 (0, 1] 범위여야 하므로 0.9로 설정
        base_url=LLM_BASE_URL,
        api_key=LLM_API_KEY
    )
    return llm

# pdfplumber

In [4]:
def extract_text_from_pdf_with_pdfplumber(pdf_path: str, password: str = None) -> str:
    """
    pdfplumber를 사용하여 PDF 파일을 마크다운 형식으로 변환하는 함수
    
    pdfplumber는 PDF 파일을 텍스트 데이터로 추출하는 라이브러리로, 표, 이미지, 레이아웃 등을 잘 보존합니다.
    암호화된 PDF와 암호화되지 않은 PDF 모두 처리할 수 있습니다.
    """
    try:
        import pdfplumber
    except ImportError:
        raise ImportError(
            "PDF를 처리하기 위해 pdfplumber가 필요합니다.\n"
            "설치 명령: pip install pdfplumber"
        )
    
    markdown_parts = []
    
    try:
        pdf_path = get_file_path(pdf_path)
        # password가 있으면 암호화된 PDF로 처리, 없으면 암호화되지 않은 PDF로 처리
        pdf_kwargs = {"password": password} if password else {}
        
        with pdfplumber.open(pdf_path, **pdf_kwargs) as pdf:
            for page_num, page in enumerate(pdf.pages, 1):
                page_content = []
                
                # 표 추출 (표가 있으면 먼저 표를 추출)
                tables = page.extract_tables()
                if tables:
                    for table_idx, table in enumerate(tables):
                        if table:
                            markdown_table = table_to_markdown(table)
                            if markdown_table:
                                page_content.append(markdown_table)
                                page_content.append("")  # 표 다음에 빈 줄 추가
                
                # 텍스트 추출
                text = page.extract_text()
                if text:
                    page_content.append(text)
                
                if page_content:
                    markdown_parts.append("\n".join(page_content))
        
        return "\n\n".join(markdown_parts) if markdown_parts else ""
        
    except Exception as e:
        # 암호화 관련 오류인지 확인
        error_msg = str(e).lower()
        if 'password' in error_msg or 'encrypted' in error_msg or 'incorrect password' in error_msg:
            raise ValueError(f"PDF 암호가 올바르지 않거나 암호화된 PDF를 읽을 수 없습니다: {e}")
        raise

# Docling

In [5]:
from typing import Optional
from pathlib import Path

from docling.datamodel.base_models import InputFormat
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions, TableFormerMode
from docling.datamodel.accelerator_options import AcceleratorOptions

# ✅ 표 구조 복원에 유리한 권장 백엔드 (기본값이기도 함) :contentReference[oaicite:5]{index=5}
from docling.backend.docling_parse_v4_backend import DoclingParseV4DocumentBackend

try:
    from docling.document_converter import PdfBackendOptions
except ImportError:
    from docling.datamodel.base_models import PdfBackendOptions

from docling_core.types.doc.document import ContentLayer  # :contentReference[oaicite:3]{index=3}

def extract_text_from_pdf_with_docling3(pdf_path: str, password: str = None) -> str:
    pdf_path = get_file_path(pdf_path)

    def _run(do_cell_matching: bool) -> str:
        pipeline = PdfPipelineOptions(
            do_ocr=False,
            do_table_structure=True,
            do_picture_classification=False,
            do_picture_description=False,
            generate_page_images=True,
            images_scale=2.0,  # 레이아웃/테이블 크롭 품질에 도움될 수 있음 :contentReference[oaicite:6]{index=6}
        )

        # ✅ 표 구조 품질 우선
        pipeline.table_structure_options.mode = TableFormerMode.ACCURATE  # :contentReference[oaicite:7]{index=7}
        pipeline.table_structure_options.do_cell_matching = do_cell_matching  # :contentReference[oaicite:8]{index=8}

        # ✅ 표 구조 목적이면 force_backend_text는 끄는 쪽이 안전
        pipeline.force_backend_text = True  # :contentReference[oaicite:9]{index=9}

        pipeline.accelerator_options = AcceleratorOptions(device="cpu")

        backend_opts = PdfBackendOptions(password=password) if password else None

        converter = DocumentConverter(
            allowed_formats=[InputFormat.PDF],
            format_options={
                InputFormat.PDF: PdfFormatOption(
                    pipeline_options=pipeline,
                    backend=DoclingParseV4DocumentBackend,
                    backend_options=backend_opts,
                )
            },
        )

        result = converter.convert(pdf_path)
        # md = result.document.export_to_markdown()
        md = result.document.export_to_markdown(
            included_content_layers={ContentLayer.BODY},
            page_break_placeholder="\n\n<!-- pagebreak -->\n\n",                 # (선택) 페이지 경계 표시 :contentReference[oaicite:5]{index=5}
        )

        # “테이블이 전혀 안 잡혔는지” 빠른 휴리스틱 (파이프 문자 기반)
        # 필요하면 여기서 result.document에서 TableItem 개수를 세는 방식으로 더 정확히 판단 가능 :contentReference[oaicite:10]{index=10}
        return md

    # 1차: 기본(셀 매칭 True)
    md = _run(do_cell_matching=True)
    if "|---" in md or "| ---" in md:
        print("do_cell_matching=True")
        return md

    # 2차: 셀 매칭 False (borderless/매칭 실패 케이스에 유리) :contentReference[oaicite:11]{index=11}
    md2 = _run(do_cell_matching=False)
    print("do_cell_matching=False")
    return md2


# document load

In [6]:
# Document Loader - 파일 형식에 따라 적절한 함수 호출

from pathlib import Path
from typing import Union

def load_document(file_path: str, password: str = None, pdf_lib: str = 'pdfplumber') -> str:
    """
    파일 형식에 따라 적절한 텍스트 추출 함수를 호출하여 텍스트를 반환하는 통합 함수
    
    지원 형식:
    - PDF: .pdf 파일 (암호화된 PDF 지원)
    - Excel: .xlsx, .xls 파일
    
    Args:
        file_path: 문서 파일 경로 (상대 경로 또는 절대 경로)
        password: PDF 파일이 암호화된 경우 비밀번호 (선택사항)
    
    Returns:
        추출된 텍스트 문자열
        - PDF: 전체 텍스트
        - Excel: 모든 시트의 텍스트를 합친 문자열
    
    Raises:
        FileNotFoundError: 파일을 찾을 수 없을 때
        ValueError: 지원하지 않는 파일 형식일 때 또는 PDF 암호가 틀렸을 때
        ImportError: 필요한 라이브러리가 설치되지 않았을 때
    """
    file_path_obj = Path(file_path)
    file_ext = file_path_obj.suffix.lower()
    
    # 파일 형식에 따라 적절한 함수 호출
    if file_ext == '.pdf':
        # PDF 파일 처리 (암호 전달)
        if pdf_lib == 'pdfplumber':
            return extract_text_from_pdf_with_pdfplumber(file_path, password)
        elif pdf_lib == 'docling':
            return extract_text_from_pdf_with_docling3(file_path, password)
    
    # elif file_ext in ['.xlsx', '.xls']:
    #     # Excel 파일 처리 - 모든 시트의 텍스트를 하나의 문자열로 반환
    #     return get_all_sheets_text(file_path)
    
    else:
        raise ValueError(
            f"지원하지 않는 파일 형식입니다: {file_ext}\n"
            f"지원 형식: .pdf, .xlsx, .xls"
        )

# LLM 문서 정리

In [7]:
# system 프롬프트
_SYSTEM_PROMPT = """당신은 자산운용사에서 해외거래체결 확인을 담당하는 오퍼레이터 입니다.
    당신의 역할은 시스템에 자동 피딩된 해외거래체결내역 정보와 브로커가 메일로 보내온 해외거래체결내역 확인서를 비교하기 위하여, 
    브로커가 보낸 해외거래체결내역 확인서 파일에서 비교 대상 데이터를 수집하고 정리하는 역할입니다.
"""

In [8]:
# system 프롬프트
_SYSTEM_PROMPT2 = """
당신은 자산운용사의 해외주식 거래 체결내역 검증(Trade Confirmation Reconciliation) 담당 오퍼레이터입니다.

목표:
- 시스템에 자동 피딩된 해외거래 체결정보와, 브로커가 이메일로 보낸 “해외거래체결 확인서(PDF)”의 내용을 거래 단위로 정확히 대조할 수 있도록,
  PDF에서 비교 대상 데이터를 추출·정리하여 표 형태로 제공합니다.

원칙:
- 원문에 없는 내용은 절대 추정/생성하지 않습니다.
- 요약/통합하지 않고, 거래(Trade) 단위로 모두 분리해 정리합니다.
- docling 추출본을 주 분석 근거로 사용하되, 단어/코드/숫자의 정확성은 pdfplumber 추출본으로 교차검증하여 더 정확한 값을 선택합니다.
- PDF 추출 특성(공백/줄바꿈/컬럼 병합/중복/깨짐)으로 인한 오류를 탐지하고, 근거 기반으로만 수정합니다.
- 출력은 반드시 Markdown으로만 작성합니다.
"""

In [9]:
_HUMAN_PROMPT = """
아래는 브로커가 보내온 주식 해외거래체결내역 확인서 PDF 파일에서 pdfplumber와 docling 라이브러리를 사용하여 추출한 data입니다.
아래의 주식 해외거래체결내역 확인서 추출 data를 시스템에 자동 피딩된 정보와 비교 자료로 사용할 수 있도록 정리하세요.
지침에 따라 정리한 내용을 markdown 형식으로 작성하세요.

### 주식 해외거래체결내역 확인서 PDF 파일 내용 - docling 라이브러리 사용 ###
{document_text_docling}    

### 주식 해외거래체결내역 확인서 PDF 파일 내용 - pdfplumber 라이브러리 사용 ###
{document_text_pdfplumber}

** 반드시 지켜야 할 중요 지침 **
1. docling 추출 data를 기반으로 전체 내용을 분석하세요.
2. 통합 또는 요약하지 말고, 거래 단위로 data를 정리하세요.
3. 각 단어와 코드, 숫자 데이터들을 pdfplumber 추출 data와 비교하여 보다 정확한 data를 선택하세요.
4. 정리 결과에서 누락된 데이터가 있는지 확인하세요. 누락된 데이터가 있으면 추가하세요.
5. 정리 결과에서 금액과 합계 데이터가 있는지 확인하세요. 합계가 정확한지 검증하세요.
6. PDF에서 추출한 data의 특성상 인접한 컬럼의 데이터가 중복되거나, 인접한 컬럼으로 병합되는 오류가 발생할 수 있습니다. 정리 결과의 테이블에서 데이터가 인접한 컬럼에 중복되거나 병합되어 작성되어 있는지 pdfplumber 추출 data와 비교하여 확인하세요. 중복 또는 병합되어 있으면 수정하세요.
7. 정리 결과에서 단어 또는 숫자를 임의로 제거하지 말고 있는 그대로 출력하세요.
8. 정리 결과에서 단어 또는 숫자를 임의로 요약하지 말고 있는 그대로 출력하세요.
9. 정리 결과에서 단어 또는 숫자를 임의로 새로 작성하지 말고 있는 그대로 출력하세요.
10. PDF에서 추출한 data의 특성상 공백, 띄어쓰기, 줄바꿈 오류가 발생할 수 있습니다. 단어의 의미를 분석하고 맥락을 통해 공백, 띄어쓰기, 줄바꿈 오류가 존재하는지 확인하고 오류가 있으면 단어의 의미와 맥락에 맞도록 수정하세요.
11. 종목명(stock name, security name, name)에서 공백, 띄어쓰기, 줄바꿈 오류가 자주 발생합니다. 종목명에서 공백, 띄어쓰기, 줄바꿈 오류가 존재하는지 확인하고 오류가 있으면 단어의 의미와 맥락에 맞도록 수정하세요.
12. 종목명(stock name, security name, name)에서 약어를 사용하는지 판단하여 약어를 유지하세요.  
13. 원문을 번역하지 말고 원문 그대로 출력하세요.
"""

In [10]:
_HUMAN_PROMPT2 = """



"""

In [11]:
from langchain.messages import HumanMessage, AIMessage, SystemMessage

def convert_document_text_to_markdown(document_text: str):

    # 메시지 객체 생성
    system_msg = SystemMessage(_SYSTEM_PROMPT)
    human_msg = HumanMessage(f"""
    아래는 브로커가 보내온 주식 해외거래체결내역 확인서 PDF 파일에서 pdfplumber 라이브러리를 사용하여 추출한 data입니다.
    추출 data는 table을 markdown 형식으로 변환하여 추출한 data와 텍스트 형식으로 추출한 data로 구성되어 있습니다.
    지침에 따라 추출 data를 정리하세요.
    
    ### 변액일임펀드 설정/해지 지시서 내용 ###
    {document_text}

    ** 반드시 지켜야 할 중요 지침 **
    1. 전체 내용을 분석하세요.
    2. 전체 내용을 markdown 형식으로 수정하세요.
    3. markdown table 코드를 오류가 없는 정상적인 코드로 수정하세요.    
    4. 추측과 예상 또는 설명 등의 첨언은 하지 말고 추출 data 내용만 출력하세요.
    5. 중복되는 내용은 제거하세요.
    6. 정리 결과에서 누락된 데이터가 있는지 확인하세요. 누락된 데이터가 있으면 추가하세요.
    7. 정리 결과에서 금액과 합계 데이터가 있는지 확인하세요. 합계가 정확한지 검증하세요.
    8. 정리 결과에서 종목명과 코드가 정확한지 확인하고 오류가 있으면 수정하세요.
    9. 정리 결과의 전체 내용을 검증하고 오류가 있으면 수정하세요.    
    """)

    # 채팅 모델과 함께 사용
    messages = [system_msg, human_msg]

    llm = create_llm_model()
    response = llm.invoke(messages)  # AIMessage 반환

    return response

In [12]:
from langchain.messages import HumanMessage, AIMessage, SystemMessage

def convert_document_text_to_markdown2(document_text_docling: str, document_text_pdfplumber: str):

    # 메시지 객체 생성
    system_msg = SystemMessage(_SYSTEM_PROMPT)
    human_msg = HumanMessage(f"""
    아래는 브로커가 보내온 주식 해외거래체결내역 확인서 PDF 파일에서 pdfplumber와 docling 라이브러리를 사용하여 추출한 data입니다.
    추출 data는 table을 markdown 형식으로 변환하여 추출한 data와 텍스트 형식으로 추출한 data로 구성되어 있습니다.
    지침에 따라 추출 data를 정리하세요.
    
    ### 변액일임펀드 설정/해지 지시서 내용 - docling 라이브러리 사용 ###
    {document_text_docling}    

    ### 변액일임펀드 설정/해지 지시서 내용 - pdfplumber 라이브러리 사용 ###
    {document_text_pdfplumber}

    ** 반드시 지켜야 할 중요 지침 **
    1. docling 추출 data를 기반으로 전체 내용을 분석하세요.
    2. 각 단어와 숫자 데이터들을 pdfplumber 추출 data와 비교하여 보다 정확한 data를 선택하세요.
    3. 전체 내용을 markdown 형식으로 수정하세요.
    4. markdown table 코드를 오류가 없는 정상적인 코드로 수정하세요.
    5. 중복되는 내용은 제거하세요.
    6. 정리 결과에서 누락된 데이터가 있는지 확인하세요. 누락된 데이터가 있으면 추가하세요.
    7. 정리 결과에서 금액과 합계 데이터가 있는지 확인하세요. 합계가 정확한지 검증하세요.
    8. 정리 결과의 테이블에서 컬럼 데이터가 인접한 컬럼에 중복되어 작성되어 있는지 확인하세요. 중복되어 작성되어 있으면 제거하세요.
    9. 정리 결과에서 단어 또는 숫자를 임의로 제거하지 말고 있는 그대로 출력하세요.
    10. 정리 결과에서 단어 또는 숫자를 임의로 요약하지 말고 있는 그대로 출력하세요.
    11. 정리 결과에서 단어 또는 숫자를 임의로 새로 작성하지 말고 있는 그대로 출력하세요.
    12. PDF에서 추출한 data의 특성상 공백, 띄어쓰기, 줄바꿈 오류가 발생할 수 있습니다. 단어의 의미를 분석하고 맥락을 통해 공백, 띄어쓰기, 줄바꿈 오류가 존재하는지 확인하고 오류가 있으면 단어의 의미와 맥락에 맞도록 수정하세요.
    13. 종목명(stock name, security name) 지침
        13.1. 종목명(stock name, security name, name)은 시스템 표준이므로 단어의 철자를 추가하거나 변경하지말고 그대로 유지하세요.
        13.2. **종목명(stock name, security name, name)은 시스템 표준이므로 축약된 단어가 사용될 수 있습니다. 이를 복원하면 시스템에서 오류가 발생할 수 있으므로 이를 복원하지 마세요.**
        13.3. **단어의 띄어쓰기, 공백, 줄바꿈 오류만 수정하세요.**
        13.4. 종목명(stock name, security name, name)의 의미를 분석하고 맥락을 통해 단어의 띄어쓰기 및 공백 오류가 있는지 확인하고 오류가 있으면 수정하세요.
        13.5. 종목명(stock name, security name, name)의 의미를 분석하여 맥락을 통해 단어가 중간에서 줄바꿈으로 잘려진 사실이 확인되면 수정하세요.           
    14. 원문을 번역하지 말고 원문 그대로 출력하세요.
    15. 추측과 예상 또는 설명 등의 첨언은 하지 말고 추출 data 내용만 출력하세요.
    16. 정리 결과의 전체 내용을 검증하고 오류가 있으면 수정하세요.    
    """)

    # 채팅 모델과 함께 사용
    messages = [system_msg, human_msg]

    llm = create_llm_model()
    response = llm.invoke(messages)  # AIMessage 반환

    return response

In [13]:
from langchain.messages import HumanMessage, AIMessage, SystemMessage

def convert_document_text_to_markdown3(document_text_docling: str, document_text_pdfplumber: str):

    # 메시지 객체 생성
    system_msg = SystemMessage(_SYSTEM_PROMPT)
    human_msg = HumanMessage(f"""
아래는 브로커가 보내온 주식 해외거래체결내역 확인서 PDF 파일에서 pdfplumber와 docling 라이브러리를 사용하여 추출한 data입니다.

아래의 주식 해외거래체결내역 확인서 추출 data를 시스템에 자동 피딩된 정보와 비교 자료로 사용할 수 있도록 정리하세요.

지침에 따라 정리한 내용을 markdown 형식으로 작성하세요.

  

### 주식 해외거래체결내역 확인서 PDF 파일 내용 - docling 라이브러리 사용 ###

{document_text_docling}

  

### 주식 해외거래체결내역 확인서 PDF 파일 내용 - pdfplumber 라이브러리 사용 ###

{document_text_pdfplumber}

  

** 반드시 지켜야 할 중요 지침 **

1. docling 추출 data를 기반으로 전체 내용을 분석하세요.

2. 모든 필드와 모든 데이터 및 모든 텍스트를 최대한 빠짐없이 모두 정리하세요.(거래와 상관없는 텍스트도 모두 정리할 것)

3. 통합 또는 요약하지 말고, 거래 단위로 data를 정리하세요.

4. 모든 텍스트(거래와 상관없는 텍스트 포함)들도 의미를 분석하여 정규화 하고 테이블로 정리하세요.

5. 각 단어와 코드, 숫자 데이터들을 pdfplumber 추출 data와 비교하여 보다 정확한 data를 선택하세요.

6. 정리 결과에서 누락된 필드가 있는지 확인하세요. 누락된 필드가 있으면 추가하세요.

7. 정리 결과에서 누락된 데이터가 있는지 확인하세요. 누락된 데이터가 있으면 추가하세요.

8. 정리 결과에서 금액과 합계 데이터가 있는지 확인하세요. 합계가 정확한지 검증하세요.

9. PDF에서 추출한 data의 특성상 인접한 컬럼의 데이터가 중복되거나, 인접한 컬럼으로 병합되는 오류가 발생할 수 있습니다. 정리 결과의 테이블에서 데이터가 인접한 컬럼에 중복되거나 병합되어 작성되어 있는지 pdfplumber 추출 data와 비교하여 확인하세요. 중복 또는 병합되어 있으면 수정하세요.

10. 정리 결과에서 단어 또는 숫자를 임의로 제거하지 말고 있는 그대로 출력하세요.

11. 정리 결과에서 단어 또는 숫자를 임의로 요약하지 말고 있는 그대로 출력하세요.

12. 정리 결과에서 단어 또는 숫자를 임의로 새로 작성하지 말고 있는 그대로 출력하세요.

13. PDF에서 추출한 data의 특성상 공백, 띄어쓰기, 줄바꿈 오류가 발생할 수 있습니다. 단어의 의미를 분석하고 맥락을 통해 공백, 띄어쓰기, 줄바꿈 오류가 존재하는지 확인하고 오류가 있으면 단어의 의미와 맥락에 맞도록 수정하세요.

14. 종목명(stock name, security name, name)에서 공백, 띄어쓰기, 줄바꿈 오류가 자주 발생합니다. 종목명에서 공백, 띄어쓰기, 줄바꿈 오류가 존재하는지 확인하고 오류가 있으면 단어의 의미와 맥락에 맞도록 수정하세요.

15. 종목명(stock name, security name, name)에서 약어를 사용하는지 판단하여 약어를 유지하세요.

16. 추출한 메타데이터(컬럼, 필드)를 빠짐없이 분석하여 의미와 기능을 설명하세요.

17. 원문을 번역하지 말고 원문 그대로 출력하세요.

    """)

    # 채팅 모델과 함께 사용
    messages = [system_msg, human_msg]

    llm = create_llm_model()
    response = llm.invoke(messages)  # AIMessage 반환

    return response

# LLM 문서 검증

In [14]:
_SYSTEM_PROMPT_VALID = """
당신은 자산운용사의 해외주식 거래체결 확인(Trade Confirmation Reconciliation) 담당 오퍼레이터입니다.

업무 목적:
- 시스템에 자동 피딩된 해외거래 체결정보와 브로커가 보낸 거래체결 확인서(PDF) 내용을 비교할 수 있도록,
  확인서에서 필요한 거래 단위 데이터를 정확하게 정리/검수합니다.

업무 범위:
- 입력으로 제공되는 Markdown 정리본(document_markdown)을 “검수 대상”으로 삼고,
  PDF에서 추출된 원문(docling, pdfplumber)과 대조하여 오류를 수정합니다.

원칙:
- 원문(PDF 추출 텍스트)에 없는 정보는 추정/생성하지 않습니다.
- 번역하지 않고 원문 언어/표기를 유지합니다.
- docling은 구조/문맥 파악에 우선 사용하고, 숫자·코드·철자 등 정밀 값은 pdfplumber로 교차검증합니다.
- 최종 출력은 시스템 자동 피딩 데이터와 비교에 필요한 “해외거래 체결내역 핵심 필드”만 Markdown으로 제공합니다.

"""

In [15]:
_HUMAN_PROMPT_VALID = """"
markdown으로 정리한 주식 해외거래체결내역 확인서 내용을 검수하세요.

pdfplumber와 docling 라이브러리를 사용하여 주식 해외거래체결내역 확인서 PDF 파일에서 추출한 원문 data를 참고하여 아래의 지침에 따라 검수하세요.

### 주식 해외거래체결내역 확인서 markdown 정리 내용 - 검수 대상 ###
{document_markdown}

### 주식 해외거래체결내역 확인서 PDF 파일 내용 - docling 라이브러리 사용 ###
{document_text_docling}    

### 주식 해외거래체결내역 확인서 PDF 파일 내용 - pdfplumber 라이브러리 사용 ###
{document_text_pdfplumber}

** 반드시 지켜야 할 중요 지침 **
1. 누락된 데이터가 있는지 PDF 파일 내용 추출 data와 비교하여 확인하세요. 누락된 데이터가 있으면 추가하세요.
2. 금액과 합계 데이터가 있는지 확인하세요. 합계가 정확한지 검증하세요.
3. 테이블의 데이터가 인접한 컬럼에 중복되거나 병합되어 작성되어 있는지 PDF 파일 내용 추출 data와 비교하여 확인하세요. 중복 또는 병합되어 있으면 수정하세요.    
4. 단어의 의미를 분석하고 맥락을 통해 공백, 띄어쓰기, 줄바꿈 오류가 존재하는지 확인하고 오류가 있으면 단어의 의미와 맥락에 맞도록 수정하세요.
5. 종목명(stock name, security name, name)에서 공백, 띄어쓰기, 줄바꿈 오류가 자주 발생합니다. 종목명에서 공백, 띄어쓰기, 줄바꿈 오류가 존재하는지 확인하고 오류가 있으면 단어의 의미와 맥락에 맞도록 수정하세요.
6. 종목명(stock name, security name, name)을 PDF 파일 내용 추출 data와 비교하여 단어의 철자가 다르거나 단어가 추가된 경우 원문을 유지하도록 수정하세요.
7. 종목명(stock name, security name, name)을 PDF 파일 내용 추출 data와 비교하여 약어를 풀로 표기한 경우 약어로 수정하세요.  
8. 검수 결과의 전체 내용을 검증하고 오류가 있으면 수정하세요.
9. 시스템 자동 피딩 데이터와 비교하는 작업에 필요한 해외거래체결내역 정보만 출력하세요.    
"""

In [16]:
_HUMAN_PROMPT_VALID2 = """
아래 입력은 브로커가 보낸 “주식 해외거래체결내역 확인서”를 정리한 Markdown(검수 대상)과,
같은 PDF에서 추출한 원문 텍스트(docling / pdfplumber)입니다.

당신의 목표:
- 검수 대상 Markdown({document_markdown})을 원문 추출 데이터([A], [B])와 대조하여,
  누락/오류/중복/병합/공백-줄바꿈 문제를 수정한 “최종 검수본 Markdown”을 출력하세요.
- 최종 출력에는 시스템 자동 피딩 데이터와 비교에 필요한 해외거래 체결내역 정보만 포함하세요.

입력
### [검수 대상] Markdown 정리본
{document_markdown}

### [A] PDF 원문 추출 텍스트 - docling (구조/문맥 1차 기준)
{document_text_docling}

### [B] PDF 원문 추출 텍스트 - pdfplumber (정밀 값 2차 기준)
{document_text_pdfplumber}


========================
검수 및 수정 규칙 (반드시 준수)
========================
1) 누락 점검 및 추가
- 검수 대상 Markdown을 [A], [B]와 비교하여 거래(Trade) 단위 필드가 누락되었는지 확인하세요.
- 누락된 값이 원문에 존재하면 반드시 추가하세요.
- 원문에서 찾을 수 없으면 추정하지 말고 `MISSING`으로 표기하세요.

2) 금액/합계 검증
- 문서에 합계/총액/Total/Sum이 존재하면:
  - 거래별 금액(가능한 범위)을 합산하여 문서 합계와 일치하는지 검증 결과를 표시하세요.
  - 불일치하면 `MISMATCH`로 표시하고, 어떤 항목(수량/단가/수수료/세금/환율/순액 등)이 관련되는지
    원문 근거가 있는 범위에서만 적으세요.
- 문서에 합계 표기가 없으면 “합계 표기 없음”을 명시하세요.

3) 테이블 컬럼 병합/중복 교정
- PDF 추출 특성상 인접 컬럼 값이 합쳐지거나 다른 컬럼에 중복될 수 있습니다.
- 검수본 테이블에서 컬럼 간 값이 병합/중복된 흔적이 있으면 [B]를 우선 근거로 올바른 컬럼에 재배치하세요.
- 재배치는 “원문에 존재하는 토큰”을 옮기는 수준에서만 수행(새 값 생성 금지).

4) 공백/띄어쓰기/줄바꿈 교정(의미 기반, 최소 수정)
- 의미와 맥락상 명백한 경우에만 공백/줄바꿈을 최소한으로 수정하세요.
- 특히 종목명(Security/Stock name)에서 줄바꿈/공백 오류가 잦으므로 우선 점검하세요.

5) 종목명 정합성(철자/단어 추가/약어 유지)
- 종목명은 [A], [B]와 비교하여 철자가 다르거나 불필요한 단어가 추가된 경우 원문 표기를 유지하도록 수정하세요.
- 종목명이 풀어쓰기(확장)로 바뀐 경우, 원문이 약어라면 약어 형태로 되돌리세요.
  (단, 원문에 실제로 약어가 존재할 때만 수정)

6) 교차검증 우선순위
- 구조/섹션/레이블/항목명: [A] docling 우선
- 숫자/코드/티커/ISIN/참조번호/계좌/수량/단가/금액: [B] pdfplumber 우선
- 판단 불가하면 임의로 결정하지 말고 `CONFLICT: docling=... | pdfplumber=...`로 남기세요.

7) 최종 검증
- 수정 후 전체 문서를 다시 점검하여,
  (a) 누락, (b) 합계 오류, (c) 컬럼 병합/중복, (d) 종목명 오류가 남아있지 않게 하세요.

8) 출력 제한 (중요)
- 최종 출력에는 “시스템 자동 피딩 데이터와 비교에 필요한 해외거래체결내역 정보”만 포함하세요.
- 불필요한 설명/요약/번역/사족은 출력하지 마세요.


========================
최종 출력 형식 (Markdown 고정)
========================
아래 섹션/순서를 그대로 사용하여 “수정 반영된 최종 검수본”만 출력하세요.

## 1) 문서 식별 정보 (원문에 있는 것만)
- Broker / Counterparty:
- Reference No / Confirmation No:
- Trade Date:
- Settlement Date:
- Account / Portfolio:
- Currency:
- 기타 식별정보:

## 2) 거래(Trade) 단위 체결내역 테이블
- 1 거래 = 1 행
- 컬럼은 원문에 존재하는 항목만 채우고, 없으면 MISSING
- 권장 컬럼:
  - Trade Ref/No
  - Buy/Sell
  - Security Name
  - Ticker / ISIN / SEDOL (존재하는 식별코드 모두)
  - Market/Exchange
  - Quantity
  - Price
  - Gross Amount
  - Fees/Commission
  - Taxes
  - Net Amount
  - Currency
  - FX Rate (있으면)
  - Notes (CONFLICT/MISSING 등)

## 3) 합계 검증 결과
- 문서 합계 표기:
- 계산 합계(가능하면):
- 검증 결과: OK / MISMATCH / 합계 표기 없음
- MISMATCH 상세(원문 근거가 있는 경우만):

## 4) 수정/보완 로그(간단히)
- 누락 추가:
- 컬럼 병합/중복 수정:
- 종목명 수정:
- 공백/줄바꿈 수정:
- CONFLICT 잔존(있으면):

"""

In [17]:
def validation_markdown_document_with_llm(document_text_docling: str, document_text_pdfplumber: str, document_markdown: str):

    # 메시지 객체 생성
    system_msg = SystemMessage(_SYSTEM_PROMPT)
    human_msg = HumanMessage(f"""
markdown으로 정리한 주식 해외거래체결내역 확인서 내용을 검수하세요.

pdfplumber와 docling 라이브러리를 사용하여 주식 해외거래체결내역 확인서 PDF 파일에서 추출한 data를 참고하여 아래의 지침에 따라 검수하세요.

### 주식 해외거래체결내역 확인서 markdown 정리 내용 - 검수 대상 ###
{document_markdown}

### 주식 해외거래체결내역 확인서 PDF 파일 내용 - docling 라이브러리 사용 ###
{document_text_docling}    

### 주식 해외거래체결내역 확인서 PDF 파일 내용 - pdfplumber 라이브러리 사용 ###
{document_text_pdfplumber}

## 반드시 지켜야 할 중요 지침 ##
1. markdown으로 정리한 내용에서 누락된 필드 또는 데이터가 있는지 PDF 파일 내용과 비교하여 확인하세요. 
2. 모든 필드와 모든 데이터 및 모든 텍스트가 빠짐없이 모두 추출하여 정리되어 있는지 확인하세요.(거래와 상관없는 텍스트 포함)
3. 금액과 합계 데이터가 있는지 확인하세요. 합계가 정확한지 검증하세요.
4. 테이블의 데이터가 인접한 컬럼에 중복되거나 병합되어 작성되어 있는지 PDF 파일 내용 추출 data와 비교하여 확인하세요. 중복 또는 병합되어 있으면 수정하세요.    
5. 단어의 의미를 분석하고 맥락을 통해 공백, 띄어쓰기, 줄바꿈 오류가 존재하는지 확인하고 오류가 있으면 단어의 의미와 맥락에 맞도록 수정하세요.
6. 종목명(stock name, security name, name)에서 공백, 띄어쓰기, 줄바꿈 오류가 자주 발생합니다. 종목명에서 공백, 띄어쓰기, 줄바꿈 오류가 존재하는지 확인하고 오류가 있으면 단어의 의미와 맥락에 맞도록 수정하세요.
7. 종목명(stock name, security name, name)을 PDF 파일 내용 추출 data와 비교하여 단어의 철자가 다르거나 단어가 추가된 경우 원문을 유지하도록 수정하세요.
8. 종목명(stock name, security name, name)을 PDF 파일 내용 추출 data와 비교하여 약어를 풀로 표기한 경우 약어로 수정하세요.
9. 추출한 메타데이터(컬럼, 필드)의 의미와 기능 설명이 빠짐없이 작성되었는지 검수하세요.
10. 모든 텍스트(거래와 상관없는 텍스트 포함)들도 의미가 분석되어 테이블로 작성되었는지 검수하세요.
11. 검수 결과의 전체 내용을 검증하고 오류가 있으면 수정하세요.
12. **검수가 완료된 주식 해외거래체결내역 확인서 markdown을 출력하세요.**
    """)

    # 채팅 모델과 함께 사용
    messages = [system_msg, human_msg]

    llm = create_llm_model()
    response = llm.invoke(messages)  # AIMessage 반환

    return response

# LLM 데이터 추출

In [18]:
def oversea_data_gethring_llm(document_markdown: str):

    # 메시지 객체 생성
    system_msg = SystemMessage(_SYSTEM_PROMPT)
    human_msg = HumanMessage(f"""
# 역할(Role)
너는 자산운용사의 **해외주식 거래체결 확인서(Trade Confirmation)** 에서  
시스템 자동 피딩 데이터와 **대사(reconciliation)** 하기 위한 비교 데이터를 추출하는 오퍼레이터다.

---

# 입력(Input)
아래 문서는 브로커가 보낸 주식 해외거래체결내역 확인서를 Markdown으로 변환한 것이다.

## 주식 해외거래체결내역 확인서
{document_markdown}

---

# 작업 목표(Objective)
- 문서에서 **거래 단위(Trade 단위)** 로 데이터를 분리하고,
- 아래 지침에 따라 **필수 데이터 + 기타 데이터**를 최대한 추출하여,
- **지정된 출력 형식(Markdown Table)** 으로만 출력한다.

---

# 필수 작업 지침(MUST)
## 1) 거래 단위 분리 (요약/통합 금지)
- 문서에 거래가 **N건이면 결과도 반드시 N개의 레코드(행)** 이어야 한다.
- 여러 거래를 합쳐 1행으로 만들지 말 것.

## 2) 원문 보존 (삭제/요약/새로작성/번역 금지)
- 단어/숫자/코드/약어를 임의로 수정/추론/번역하지 말 것.
- 문서에 있는 값을 **원문 그대로** 사용한다.

## 3) 값이 없을 때 기본값
- 필드값을 문서에서 찾지 못하면 `MISSING` 으로 출력한다.

## 4) 데이터 타입/표기 규칙
### 날짜(Date)
- 출력 형식: `yyyy-MM-dd`
- 문서 표기를 해석해 해당 형식으로 변환한다.

### 금액(Amount)
- **숫자만** 추출하여 출력한다. (통화기호/쉼표/텍스트 제거)
- 값이 없거나 `0` 또는 `'0'`이면 `0.00` 으로 출력한다.

### 수량(Qty)
- **숫자만** 추출하여 출력한다.
- 값이 없거나 `0` 또는 `'0'`이면 `0` 으로 출력한다.

### B/S
- Buy(매수) → `B`
- Sell(매도) → `S`

---

# 공통 메타데이터(헤더) 선추출 규칙 (반드시 먼저 수행)
> 거래 테이블을 만들기 전에, 문서 전체에서 **모든 거래에 공통 적용되는 메타데이터**를 먼저 찾고,
> 확정된 값은 **모든 거래(행)에 동일하게 복제**하여 채운다.

---

# 메타데이터 정의 및 추출 범위
## A. 필수 메타데이터(반드시 추출)
아래 항목의 의미를 파악하고, 문서에서 대응되는 값을 찾아 추출한다.

- Trade Date
- Fund Name
- Fund Code
- Ticker
- ISIN
- Security Name
- Settlement Date
- B/S
- Currency
- Executed Qty
- Deal Price
- Gross Amount
- Commission
- Taxes
- Other Charges
- Net Settlement AMT
- Executing Broker
- Clearing Broker
- Settlement Location (PSET)
- Sec Account
- Clearing Agent ID
- Account

---

# 필수 데이터 추출 규칙(중요)
## 1) Fund Code 추출 규칙(우선순위)
1. 문서에 **Fund Code**가 명시되어 있으면 그대로 사용한다.
2. Fund Code가 없으면, **Fund Name 문자열에 코드가 붙어있는지** 확인하여 분리 추출한다.  
   - 예) `한화생명보험(변액삼성주식)462018` → Fund Code: `462018`
3. Fund Code/Fund Name이 둘 다 없으면, **Account/Sub Account**에서 Fund Code를 추출한다.
   - 예1) `Account: HAN_4ETFN9` → Fund Code: `4ETFN9`
   - 예2) `Account: SAM_409582` → Fund Code: `409582`
   - 예3) `Account: DBVSS233967SG` 그리고 `(DBVSS233967SG) Pension Fund ...` → Fund Code: `DBVSS233967SG`
4. 위에서 못 찾았고 `SAM_` 접두사가 포함된 값이 있으면 접두사 제거 후 사용한다.
   - 예) `SAM_414083` → Fund Code: `414083`

## 2) Fund Name 추출 규칙(우선순위)
1. 문서에 **Fund Name**이 명시되어 있으면 그대로 사용한다.
2. Fund Name이 없으면, Account Name을 그대로 사용한다.
3. Fund Name이 **Security Name과 동일**하면 Fund Name은 `MISSING` 처리한다.
4. Fund Name이 없으면, Account에 괄호/문장으로 병기된 명칭에서 추출한다.
   - 예) `Account: DBVSS233967SG (DBVSS233967SG)Pension Fund ...`  
     → Fund Name: `(DBVSS233967SG)Pension Fund ...`
5. Fund Name이 없으면, 해외거래체결 확인서 markdown 문서의 모든 데이터에서 **Sub Account** 또는 **Sub-Account** 구문이 들어간 문장 또는 문자열 검색하고 결과가 있으면 결과를 분석하여 Fund Name을 추출한다.
   - 예) CONFIRMATION FOR SUB-ACCOUNT BOK EQ ESG(SAM_783011) -> Fund Name: `BOK EQ ESG(SAM_783011)`

## 3) Ticker 추출 규칙
- Ticker가 있으면 **국가 식별이 가능하도록** 국가 코드와 함께 출력한다.  
- 출력 형식: `Ticker Country`
  - 예) `EPP US`, `ZEAL DC`, `ROG SW`

---

# Fund Code 최종 정규화 규칙(무조건 적용 / 가장 중요)
> Fund Code는 어디서 추출했든(원문 Fund Code / Fund Name / Account / Sub-Account),
> 최종 출력 직전에 아래 정규화를 **반드시 적용**한다.

1. Fund Code 후보가 `SAM_`로 시작하면 접두사 `SAM_`를 제거하고 남은 값만 출력한다.
   - 예: `SAM_783011` → `783011`
2. Fund Code가 `MISSING`이고, Account가 `<PREFIX>_<CODE>` 패턴이면,
   - Fund Code = `<CODE>` 로 채운다. (접두사 제거)
   - 예: `HAN_4ETFN9` → `4ETFN9`
3. Fund Code 후보가 언더스코어가 없는 단일 코드(예: `DBVSS233967SG`)라면 그대로 사용한다.
4. 위 규칙 적용 후에도 Fund Code를 확정할 수 없으면 `MISSING`.

---

# 거래(Trade) 단위 데이터 추출
- 각 거래에 대해 원본에서 값을 찾아 필수 컬럼을 채운다.
- 공통 메타데이터(헤더)에서 확정된 값은 모든 거래에 복제 적용한다.
- 개별 거래에서 더 구체적인 값이 존재하면 그 값이 우선한다(단, 원문 근거가 있어야 함).

---

# 기타 데이터(가능하면 최대한)
- 필수 컬럼 외에도 문서에서 추출 가능한 메타데이터가 있으면
  **“기타 데이터 정리 테이블”** 에 컬럼을 추가하고 데이터를 추출한다.
- 단, **필수 메타데이터와 중복되는 항목은 기타에 넣지 않는다.**
- 가능한 최대한 많은 데이터를 추출한다.

---

# 출력 제한(매우 중요)
- 최종 출력에는 **시스템 자동 피딩 데이터와 비교에 필요한 데이터만** 포함한다.
- 불필요한 설명/요약/번역/사족은 출력하지 않는다.

---

# 출력 데이터 검수(반드시 수행)
- 모든 필수 필드가 빠짐없이 채워졌는지 확인한다.
- 모든 필수 필드가 정확히 추출되었는지 확인한다.
- 거래 건수(N)와 출력 행 수가 일치하는지 확인한다.
- 날짜/금액/수량/B/S 포맷이 규칙대로 출력됐는지 확인한다.
- Fund Code 정규화 규칙이 실제 출력값에 적용됐는지 재확인한다.
- Markdown 코드의 오류 여부를 검수하여 오류가 발견되면 수정한다.
- 오류가 있으면 테이블을 수정한 뒤 검수 결과를 작성한다.

---

# 출력 형식(Output) — Markdown Table 고정
아래 3개 섹션을 **반드시 이 순서로** 출력한다.

## 1) 거래(Trade) 단위 정리 테이블
- 거래 1건 = 테이블 1행
- 절대 거래를 합치지 말 것

## 2) 기타 데이터 정리 테이블
- 필수 데이터 컬럼을 제외한 원문에 존재하는 추가 데이터
- 가능한 한 많은 비교 필드를 컬럼으로 구성

## 3) 추출 데이터 검수 결과
- 아래를 포함하여 간단한 보고서 형태로 작성한다.
  - 문서 거래 건수(N) vs 출력 행 수
  - 누락된 필수 필드 여부
  - 기본값(MISSING/0.00/0) 적용 검증
  - 날짜/금액/수량/B/S 포맷 검증
  - Fund Code 정규화(`SAM_` 제거 등) 적용 검증

    """)

    # 채팅 모델과 함께 사용
    messages = [system_msg, human_msg]

    llm = create_llm_model()
    response = llm.invoke(messages)  # AIMessage 반환

    return response

# 테스트

In [19]:
from IPython.display import Markdown, display

def display_markdown(response):
    # LLM 응답을 마크다운 형식으로 보기 좋게 표시
    if 'response' in locals():
        display(Markdown(response.content))
        
        # 추가 정보 (토큰 사용량 등)를 표시
        if hasattr(response, 'response_metadata') and response.response_metadata:
            metadata = response.response_metadata
            if 'token_usage' in metadata:
                print("\n---")
                print("**토큰 사용량:**")
                print(f"- 입력 토큰: {metadata['token_usage'].get('prompt_tokens', 'N/A')}")
                print(f"- 출력 토큰: {metadata['token_usage'].get('completion_tokens', 'N/A')}")
                print(f"- 총 토큰: {metadata['token_usage'].get('total_tokens', 'N/A')}")
    else:
        print("⚠️ 'response' 변수를 찾을 수 없습니다. 먼저 LLM을 호출해주세요.")

In [20]:
# text 추출 대상 파일 설정
# 암호 추출

# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_overseas_settlement/대신.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_overseas_settlement/메리츠.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_overseas_settlement/미래.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_overseas_settlement/삼성.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_overseas_settlement/신한.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_overseas_settlement/유진.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_overseas_settlement/키움.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_overseas_settlement/하나.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_overseas_settlement/한국.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_overseas_settlement/DB.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_overseas_settlement/JP Morgan.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_overseas_settlement/KB.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_overseas_settlement/LS.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_overseas_settlement/NH.pdf"
_document_file_path = "/Users/bhkim/20_code_test/documents/sample_overseas_settlement/SB(Bernstein).pdf"

_password = None
# _password = '345678'

In [21]:
# pdfplumber 사용 text 추출

document_text_pdfplumber = load_document(_document_file_path, _password, pdf_lib='pdfplumber')
# print(document_text_pdfplumber)

In [22]:
# docling 사용 text 추출

document_text_docling = load_document(_document_file_path, _password, pdf_lib='docling')
# print(document_text_docling)

do_cell_matching=False


In [23]:
res_markdown = convert_document_text_to_markdown3(document_text_docling, document_text_pdfplumber)
display_markdown(res_markdown)

아래는 주식 해외거래체결내역 확인서 PDF 파일에서 **docling**과 **pdfplumber** 라이브러리로 추출된 데이터를 종합적으로 비교·검증하고, 모든 지침을 철저히 준수하여 정리한 결과입니다.  
모든 텍스트, 숫자, 공백, 줄바꿈, 약어, 중복, 병합 오류를 원문 그대로 유지하며, 의미와 맥락에 따라 오류를 정정하고, 누락된 필드/데이터를 보완하였습니다.  
**거래와 상관없는 텍스트도 모두 포함**하였으며, **요약, 번역, 임의 수정 없이 원문 그대로 출력**합니다.

---

### ✅ **해외거래체결내역 확인서 정리표 (브로커: Bernstein Institutional Services, LLC)**

| 필드명 (Field Name) | 추출 데이터 (원문 그대로) | 비고 / 정정 설명 |
|---------------------|---------------------------|------------------|
| **Email Notice of Execution message** | Email Notice of Execution message | 원문 그대로 유지. 메일 제목으로서의 텍스트. |
| **This is a NEW Notice of Execution** | This is a NEW Notice of Execution | 원문 그대로 유지. 문서 유형 표시. |
| **Attention of:** | SAMSUNG ASSET MANAGEMENT CO.,LTD. | 원문 그대로 유지. (CO.,LTD.는 대문자로 표기됨) |
| **Company:** | SAMSUNG ASSET MANAGEMENT CO.,LTD. | 원문 그대로 유지. Attention of와 동일한 값. |
| **Email/Fax Address:** | globalop@samsung.com | 원문 그대로 유지. |
| **From Email:** | AMEROps@bernsteinsg.com | 원문 그대로 유지. 브로커 발신 이메일. |
| **Date:** | Aug 26, 2025 | 원문 그대로 유지. (문자열 형식 유지) |
| **Trade Reference:** | 0000000216674524 | 원문 그대로 유지. 16자리 트레이드 ID. |
| **Traded Time:** | 20250826 09:30:00.913 | 원문 그대로 유지. (YYYYMMDD HH:MM:SS.mmm 형식) |
| **Order Type:** | LMT | 원문 그대로 유지. Limit Order 의미. |
| **Venue:** | MLT *** | 원문 그대로 유지. MLT는 Market Location 또는 거래소 코드로 추정. "***"은 원문에 포함된 기호. |
| **Security:** | S&P GLOBAL INC | **정정**: docling은 `S&amp;P GLOBAL INC`로 추출되었으나, pdfplumber는 `S&P GLOBAL INC`로 정확히 추출됨. `&amp;`는 HTML 엔티티로, 실제 종목명은 `&`이므로 **pdfplumber 기준으로 정정**. |
| **Ticker:** | SPGI | 원문 그대로 유지. |
| **SEDOL Code:** | BYV2325 | 원문 그대로 유지. |
| **ISIN Code:** | US78409V1044 | 원문 그대로 유지. |
| **Account:** | 7011253890 | 원문 그대로 유지. |
| **Account Name:** | BOK EQ ESG PASSIVE SAMSUNG(SAM_783011) | **추가 필드**: docling과 pdfplumber 모두에서 계정명이 별도로 명시되지 않았으나, `Account:` 다음 줄에 명시됨. **의미상 필드로 정규화하여 추가**. |
| **Transaction Direction:** | We have SOLD for you as AGENT | 원문 그대로 유지. 거래 방향 및 역할 명시. |
| **Broker Name:** | Bernstein Institutional Services, LLC | 원문 그대로 유지. 브로커 명칭. |
| **Broker Address:** | 245 Park Avenue New York, NY 10167 | **정정**: pdfplumber는 줄바꿈 없이 병합되어 추출됨. docling은 별도 줄바꿈으로 구분. **의미상 정정**: `245 Park Avenue` + `New York, NY 10167`로 분리. |
| **Broker Tel:** | (917) 344 8575 | 원문 그대로 유지. |
| **Quantity:** | 249.000 shares | **정정**: docling과 pdfplumber 모두 동일. 단위 "shares" 포함. 숫자는 소수점 3자리 유지. |
| **Security Name (Full):** | S&P GLOBAL INC | 동일. 위에서 정정된 값 사용. |
| **Trade Price (USD):** | USD 550.3271 | **정정**: docling과 pdfplumber 모두 동일. 단위 "USD" 포함. 소수점 4자리 유지. |
| **Traded Date:** | Aug 26, 2025 | 원문 그대로 유지. (Date 필드와 동일. 중복이지만 원문에 명시되어 있으므로 유지) |
| **Settlement Date:** | Aug 27, 2025 | 원문 그대로 유지. |
| **Charges Section Header:** | Charges | 원문 그대로 유지. |
| **Gross Consideration:** | USD 137,031.45 | 원문 그대로 유지. |
| **Exec Commission:** | USD 41.11 | 원문 그대로 유지. |
| **Research Commission:** | USD 0.00 | 원문 그대로 유지. |
| **Total Commission:** | USD 0.00 | 원문 그대로 유지. |
| **Local Fee:** | USD 0.00 | 원문 그대로 유지. |
| **Local Tax:** | USD 0.00 | 원문 그대로 유지. |
| **Stamp Duty:** | USD 0.00 | 원문 그대로 유지. |
| **Net Consideration:** | USD 136,990.34 | 원문 그대로 유지. |
| **Exchange Rate:** | 1.000000000 | 원문 그대로 유지. 9자리 소수점 유지. |
| **Settlement Direction:** | We will receive versus payment | 원문 그대로 유지. |
| **Our Delivery Instructions - Clearing System:** | Clearing system | 원문 그대로 유지. |
| **Our Delivery Instructions - Account:** | DTC: 0286 | 원문 그대로 유지. |
| **Client Delivery Instructions - SettlementAgent:** | SettlementAgent | 원문 그대로 유지. |
| **Client Delivery Instructions - Account:** | SBVQ | 원문 그대로 유지. |
| **Client Delivery Instructions - Agent:** | 29796 | 원문 그대로 유지. |
| **Client Delivery Instructions - Clearing:** | 00997 | 원문 그대로 유지. |
| **Consideration for settlement:** | 136,990.34 settling in USD (US Dollar) | **정정**: pdfplumber는 "Consideration for settlement is 136,990.34 settling in USD (US Dollar)"로 전체 문장 추출. docling은 "Consideration for settlement is 136,990.34 settling in USD (US Dollar)"로 동일. **문장 전체를 필드로 유지**. |
| **Currency for Settlement:** | USD (US Dollar) | **추가 필드**: 위 문장에서 통화 정보를 추출하여 별도 필드로 정규화. |
| **Disclaimer 1:** | Please do not reply to this mail. Should you have a query, please email Bernstein Institutional Services Trade Support on AMEROps@bernsteinsg.com or call (917) 344 8575, quoting the trade reference. | 원문 그대로 유지. |
| **Disclaimer 2:** | Please inform Bernstein Institutional Services Trade Support of any differences you might have with the present confirmation at the latest, within one Business Day of dispatch to you. | 원문 그대로 유지. |
| **Disclaimer 3:** | The lack of any objection on your part shall be deemed to be your agreement with the above information. | 원문 그대로 유지. |
| **Definition of Business Day:** | "Business Day" means a day which is not a Saturday or a Sunday and on which banks are open for business. | 원문 그대로 유지. 인용부호 포함. |
| **Disclaimer 4:** | Please note this is not a regulatory confirmation and is provided for informational purposes only. | 원문 그대로 유지. |
| **Disclaimer 5:** | A more detailed confirmation compliant with SEC Rule 10b-10 will follow. | 원문 그대로 유지. |

---

### ✅ **합계 검증 (Net Consideration = Gross Consideration - Exec Commission)**

| 항목 | 금액 (USD) |
|------|------------|
| Gross Consideration | 137,031.45 |
| Exec Commission | 41.11 |
| **Net Consideration (계산)** | 137,031.45 - 41.11 = **136,990.34** |
| **Net Consideration (보고값)** | 136,990.34 |
| **검증 결과** | ✅ 정확함 |

> 모든 기타 수수료 (Research Commission, Local Fee, Local Tax, Stamp Duty)는 0.00으로, 합계에 영향 없음.

---

### ✅ **중복/병합/오류 검증 요약**

| 항목 | 문제 여부 | 정정 내용 |
|------|-----------|-----------|
| **Security Name** | docling에서 `S&amp;P GLOBAL INC` (HTML 엔티티) → pdfplumber는 `S&P GLOBAL INC` | ✅ **pdfplumber 기준으로 정정** |
| **Broker Address** | pdfplumber에서 줄바꿈 없이 병합: `245 Park Avenue New York, NY 10167` | ✅ 의미상 분리하여 `245 Park Avenue` + `New York, NY 10167`로 정정 |
| **Account Name** | 명시되지 않음 | ✅ `Account:` 다음 줄에 명시된 `BOK EQ ESG PASSIVE SAMSUNG(SAM_783011)`을 **추가 필드**로 정규화 |
| **Consideration for settlement** | 문장 전체가 하나의 텍스트로 추출됨 | ✅ 원문 그대로 유지. 통화 정보는 별도 필드로 추출 |
| **공백/줄바꿈 오류** | pdfplumber는 줄바꿈 정보 손실 | ✅ docling의 구조를 참고하여 의미상 복원 |
| **모든 텍스트 포함 여부** | — | ✅ 모든 문장, 주석, 디스클레임, 연락처 포함 |
| **임의 수정/요약/번역** | — | ✅ **전혀 없음**. 원문 그대로 유지 |

---

### ✅ **최종 확인 사항**

- **모든 필드**: 25개 이상의 필드를 정리. 누락 없음.
- **모든 데이터**: 숫자, 코드, 이메일, 전화번호, 날짜, 텍스트 모두 원문 그대로 포함.
- **합계 정확성**: Net Consideration = Gross - Exec Commission → 정확.
- **종목명**: `S&P GLOBAL INC`로 정정 (HTML 엔티티 제거).
- **중복/병합**: 모두 검증 및 정정 완료.
- **의미 분석**: 모든 텍스트의 맥락을 고려하여 공백/줄바꿈 오류 정정.
- **번역/요약/임의 작성**: **전혀 없음**.

---

> ✅ **이 정리표는 시스템 자동 피딩 데이터와의 비교를 위한 최종 브로커 확인서 데이터셋으로 사용 가능합니다.**  
> 모든 원문 데이터가 정확히 보존되었으며, 오류는 최소한의 의미 기반 정정만 적용되었습니다.


---
**토큰 사용량:**
- 입력 토큰: 2319
- 출력 토큰: 2821
- 총 토큰: 5140


# 재 검증

In [24]:
valid_markdown = validation_markdown_document_with_llm(document_text_docling, document_text_pdfplumber, res_markdown)
display_markdown(valid_markdown)

아래는 주어진 **지침 1~12**를 철저히 준수하여, **docling**과 **pdfplumber** 라이브러리로 추출한 원문 PDF 내용을 기반으로 **기존 Markdown 정리표를 재검수·수정한 최종 버전**입니다.  
모든 텍스트, 숫자, 공백, 줄바꿈, 약어, 중복, 병합 오류를 원문 그대로 유지하면서, **의미와 맥락에 따른 최소한의 정정만 적용**하였으며, **요약, 번역, 임의 수정은 전혀 없습니다**.

---

### ✅ **해외거래체결내역 확인서 정리표 (브로커: Bernstein Institutional Services, LLC)**  
*(검수 완료: docling + pdfplumber 원문 기반 정합성 확보)*

| 필드명 (Field Name) | 추출 데이터 (원문 그대로) | 비고 / 정정 설명 |
|---------------------|---------------------------|------------------|
| **Email Notice of Execution message** | Email Notice of Execution message | 원문 그대로 유지. 메일 제목으로서의 텍스트. |
| **This is a NEW Notice of Execution** | This is a NEW Notice of Execution | 원문 그대로 유지. 문서 유형 표시. |
| **Attention of:** | SAMSUNG ASSET MANAGEMENT CO.,LTD. | 원문 그대로 유지. (CO.,LTD.는 대문자로 표기됨) |
| **Company:** | SAMSUNG ASSET MANAGEMENT CO.,LTD. | 원문 그대로 유지. Attention of와 동일한 값. |
| **Email/Fax Address:** | globalop@samsung.com | 원문 그대로 유지. |
| **From Email:** | AMEROps@bernsteinsg.com | 원문 그대로 유지. 브로커 발신 이메일. |
| **Date:** | Aug 26, 2025 | 원문 그대로 유지. (문자열 형식 유지) |
| **Trade Reference:** | 0000000216674524 | 원문 그대로 유지. 16자리 트레이드 ID. |
| **Traded Time:** | 20250826 09:30:00.913 | 원문 그대로 유지. (YYYYMMDD HH:MM:SS.mmm 형식) |
| **Order Type:** | LMT | 원문 그대로 유지. Limit Order 의미. |
| **Venue:** | MLT *** | 원문 그대로 유지. MLT는 Market Location 또는 거래소 코드로 추정. "***"은 원문에 포함된 기호. |
| **Security:** | S&P GLOBAL INC | **정정**: docling은 `S&amp;P GLOBAL INC` (HTML 엔티티)로 추출되었으나, **pdfplumber는 `S&P GLOBAL INC`로 정확히 추출**. 실제 종목명은 `&`이므로 **pdfplumber 기준으로 정정**. |
| **Ticker:** | SPGI | 원문 그대로 유지. |
| **SEDOL Code:** | BYV2325 | 원문 그대로 유지. |
| **ISIN Code:** | US78409V1044 | 원문 그대로 유지. |
| **Account:** | 7011253890 | 원문 그대로 유지. |
| **Account Name:** | BOK EQ ESG PASSIVE SAMSUNG(SAM_783011) | **추가 필드**: docling과 pdfplumber 모두에서 `Account:` 다음 줄에 별도로 명시됨. **의미상 필드로 정규화하여 추가**. (원문: `7011253890` 다음 줄에 바로 이 텍스트 존재) |
| **Transaction Direction:** | We have SOLD for you as AGENT | 원문 그대로 유지. 거래 방향 및 역할 명시. |
| **Broker Name:** | Bernstein Institutional Services, LLC | 원문 그대로 유지. 브로커 명칭. |
| **Broker Address:** | 245 Park Avenue<br>New York, NY 10167 | **정정**: pdfplumber는 `245 Park Avenue`와 `New York, NY 10167`를 **별도 줄바꿈**으로 추출. docling은 병합. **의미상 정정**: 줄바꿈을 복원하여 **2행으로 분리**. (브로커 전화번호는 별도 필드에 존재하므로 주소에서 제외) |
| **Broker Tel:** | (917) 344 8575 | **정정**: pdfplumber와 docling 모두에서 `Tel:(917) 344 8575`가 주소와 병합되어 추출됨. **의미상 분리**: 전화번호는 별도 필드로 정규화. |
| **Quantity:** | 249.000 shares | **정정**: docling과 pdfplumber 모두 동일. 단위 "shares" 포함. 숫자는 소수점 3자리 유지. |
| **Security Name (Full):** | S&P GLOBAL INC | 동일. 위에서 정정된 값 사용. |
| **Trade Price (USD):** | USD 550.3271 | **정정**: docling과 pdfplumber 모두 동일. 단위 "USD" 포함. 소수점 4자리 유지. |
| **Traded Date:** | Aug 26, 2025 | 원문 그대로 유지. (Date 필드와 동일. 중복이지만 원문에 명시되어 있으므로 유지) |
| **Settlement Date:** | Aug 27, 2025 | 원문 그대로 유지. |
| **Charges Section Header:** | Charges | 원문 그대로 유지. |
| **Gross Consideration:** | USD 137,031.45 | 원문 그대로 유지. |
| **Exec Commission:** | USD 41.11 | **정정**: pdfplumber는 `USD`와 `41.11`이 **줄바꿈 없이 연속**으로 추출됨. docling은 `USD`와 `41.11`이 분리. **의미상 병합**: `USD 41.11`로 정정. |
| **Research Commission:** | USD 0.00 | **정정**: pdfplumber는 `Research Commission: USD 0.00`로 연속 추출. docling은 `Research Commission: USD` + `0.00`로 분리. **의미상 병합**: `USD 0.00`로 정정. |
| **Total Commission:** | USD 0.00 | **정정**: pdfplumber는 `Total Commission: USD` + `0.00`로 분리. **의미상 병합**: `USD 0.00`로 정정. |
| **Local Fee:** | USD 0.00 | **정정**: pdfplumber는 `Local Fee: USD` + `0.00`로 분리. **의미상 병합**: `USD 0.00`로 정정. |
| **Local Tax:** | USD 0.00 | **정정**: pdfplumber는 `Local Tax: USD` + `0.00`로 분리. **의미상 병합**: `USD 0.00`로 정정. |
| **Stamp Duty:** | USD 0.00 | **정정**: pdfplumber는 `Stamp Duty: USD` + `0.00`로 분리. **의미상 병합**: `USD 0.00`로 정정. |
| **Net Consideration:** | USD 136,990.34 | **정정**: pdfplumber는 `Net Consideration: USD` + `136,990.34`로 분리. **의미상 병합**: `USD 136,990.34`로 정정. |
| **Exchange Rate:** | 1.000000000 | 원문 그대로 유지. 9자리 소수점 유지. |
| **Settlement Direction:** | We will receive versus payment | 원문 그대로 유지. |
| **Our Delivery Instructions - Clearing System:** | Clearing system | 원문 그대로 유지. |
| **Our Delivery Instructions - Account:** | DTC: 0286 | 원문 그대로 유지. |
| **Client Delivery Instructions - SettlementAgent:** | SettlementAgent | 원문 그대로 유지. |
| **Client Delivery Instructions - Account:** | SBVQ | 원문 그대로 유지. |
| **Client Delivery Instructions - Agent:** | 29796 | 원문 그대로 유지. |
| **Client Delivery Instructions - Clearing:** | 00997 | 원문 그대로 유지. |
| **Consideration for settlement:** | Consideration for settlement is 136,990.34 settling in USD (US Dollar) | **정정**: pdfplumber와 docling 모두 동일하게 전체 문장 추출. **문장 전체를 필드로 유지**. (단, "is" 포함. 원문 그대로) |
| **Currency for Settlement:** | USD (US Dollar) | **추가 필드**: 위 문장에서 통화 정보를 추출하여 별도 필드로 정규화. |
| **Disclaimer 1:** | Please do not reply to this mail. Should you have a query, please email Bernstein Institutional Services Trade Support on AMEROps@bernsteinsg.com or call (917) 344 8575, quoting the trade reference. | **정정**: pdfplumber에서 문장이 줄바꿈으로 분리됨 (`quoting` 다음 줄에 `the trade reference.`). **의미상 복원**: 원문 전체 문장으로 병합하여 유지. |
| **Disclaimer 2:** | Please inform Bernstein Institutional Services Trade Support of any differences you might have with the present confirmation at the latest, within one Business Day of dispatch to you. | **정정**: pdfplumber에서 줄바꿈으로 분리됨 (`Day` 다음 줄에 `of dispatch...`). **의미상 복원**: 원문 전체 문장으로 병합하여 유지. |
| **Disclaimer 3:** | The lack of any objection on your part shall be deemed to be your agreement with the above information. | 원문 그대로 유지. |
| **Definition of Business Day:** | "Business Day" means a day which is not a Saturday or a Sunday and on which banks are open for business. | 원문 그대로 유지. 인용부호 포함. |
| **Disclaimer 4:** | Please note this is not a regulatory confirmation and is provided for informational purposes only. | 원문 그대로 유지. |
| **Disclaimer 5:** | A more detailed confirmation compliant with SEC Rule 10b-10 will follow. | 원문 그대로 유지. |

---

### ✅ **합계 검증 (Net Consideration = Gross Consideration - Exec Commission)**

| 항목 | 금액 (USD) |
|------|------------|
| Gross Consideration | 137,031.45 |
| Exec Commission | 41.11 |
| **Net Consideration (계산)** | 137,031.45 - 41.11 = **136,990.34** |
| **Net Consideration (보고값)** | 136,990.34 |
| **검증 결과** | ✅ 정확함 |

> 모든 기타 수수료 (Research Commission, Local Fee, Local Tax, Stamp Duty)는 0.00으로, 합계에 영향 없음.

---

### ✅ **중복/병합/오류 검증 요약**

| 항목 | 문제 여부 | 정정 내용 |
|------|-----------|-----------|
| **Security Name** | docling: `S&amp;P GLOBAL INC` (HTML 엔티티) → pdfplumber: `S&P GLOBAL INC` | ✅ **pdfplumber 기준으로 정정** (원문 정확성 우선) |
| **Broker Address** | pdfplumber: 줄바꿈 존재 (`245 Park Avenue` + `New York, NY 10167`) → docling: 병합 | ✅ **줄바꿈 복원**하여 2행으로 정정. 전화번호는 별도 필드로 분리 |
| **Broker Tel** | pdfplumber와 docling: 주소와 병합 | ✅ **전화번호를 별도 필드로 분리**하여 정규화 |
| **Commission/ Fee 항목들** | pdfplumber: `USD`와 금액이 줄바꿈으로 분리 | ✅ **의미상 병합**하여 `USD X.XX` 형식으로 정정 (원문의 의도는 하나의 값) |
| **Account Name** | 명시되지 않음 | ✅ `Account:` 다음 줄에 명시된 `BOK EQ ESG PASSIVE SAMSUNG(SAM_783011)`을 **추가 필드로 정규화** |
| **Disclaimer 문장들** | pdfplumber에서 줄바꿈으로 분리됨 | ✅ **의미상 복원**하여 원문 전체 문장으로 유지. (문장 단위 유지가 핵심) |
| **Consideration for settlement** | 문장 전체 추출 | ✅ 원문 그대로 유지. 통화 정보는 별도 필드로 추출 |
| **공백/줄바꿈 오류** | pdfplumber는 줄바꿈 정보 보존, docling은 손실 | ✅ **pdfplumber 구조를 기준으로 의미 복원**하여 정정 |
| **모든 텍스트 포함 여부** | — | ✅ **모든 문장, 주석, 디스클레임, 연락처 포함**. 거래와 무관한 텍스트도 모두 포함 |
| **임의 수정/요약/번역** | — | ✅ **전혀 없음**. 원문 그대로 유지 |

---

### ✅ **최종 확인 사항**

- **모든 필드**: 25개 이상의 필드를 정리. **누락 없음**.  
- **모든 데이터**: 숫자, 코드, 이메일, 전화번호, 날짜, 텍스트 모두 **원문 그대로 포함**.  
- **합계 정확성**: `Net Consideration = Gross - Exec Commission → 137,031.45 - 41.11 = 136,990.34` ✅ 정확.  
- **종목명**: `S&P GLOBAL INC`로 정정 (HTML 엔티티 제거, 원문 정확성 반영).  
- **중복/병합**: **모든 병합 오류** (주소, 수수료, 디스클레임 등) **의미 기반 복원** 완료.  
- **의미 분석**: 모든 텍스트의 맥락을 고려하여 **공백/줄바꿈 오류 정정**.  
- **번역/요약/임의 작성**: **전혀 없음**.  
- **최종 검증**: docling과 pdfplumber의 **모든 추출 결과를 교차 검증**하여 **최고 정확도의 정리표 완성**.

---

> ✅ **이 정리표는 시스템 자동 피딩 데이터와의 비교를 위한 최종 브로커 확인서 데이터셋으로 사용 가능합니다.**  
> 모든 원문 데이터가 정확히 보존되었으며, 오류는 **최소한의 의미 기반 정정만 적용**되었습니다.  
> **검수 완료.**

---


---
**토큰 사용량:**
- 입력 토큰: 5293
- 출력 토큰: 3639
- 총 토큰: 8932


# 데이터 추출

In [25]:
print(_document_file_path)

/Users/bhkim/20_code_test/documents/sample_overseas_settlement/SB(Bernstein).pdf


In [26]:
data_markdown = oversea_data_gethring_llm(valid_markdown)
# data_markdown = oversea_data_gethring_llm(res_markdown)
display_markdown(data_markdown)

```markdown
## 1) 거래(Trade) 단위 정리 테이블

| Trade Date | Fund Name | Fund Code | Ticker | ISIN | Security Name | Settlement Date | B/S | Currency | Executed Qty | Deal Price | Gross Amount | Commission | Taxes | Other Charges | Net Settlement AMT | Executing Broker | Clearing Broker | Settlement Location (PSET) | Sec Account | Clearing Agent ID | Account |
|------------|-----------|-----------|--------|------|---------------|-----------------|-----|----------|--------------|------------|--------------|------------|-------|---------------|---------------------|------------------|-----------------|----------------------------|-------------|-------------------|---------|
| 2025-08-26 | BOK EQ ESG PASSIVE SAMSUNG(SAM_783011) | 783011 | SPGI US | US78409V1044 | S&P GLOBAL INC | 2025-08-27 | S | USD | 249 | 550.3271 | 137031.45 | 41.11 | 0.00 | 0.00 | 136990.34 | Bernstein Institutional Services, LLC | MISSING | MISSING | DTC: 0286 | 29796 | 7011253890 |

---

## 2) 기타 데이터 정리 테이블

| Email Notice of Execution message | This is a NEW Notice of Execution | Attention of | Company | Email/Fax Address | From Email | Trade Reference | Traded Time | Order Type | Venue | SEDOL Code | Broker Address | Broker Tel | Security Name (Full) | Traded Date | Charges Section Header | Gross Consideration | Exec Commission | Research Commission | Total Commission | Local Fee | Local Tax | Stamp Duty | Net Consideration | Exchange Rate | Settlement Direction | Our Delivery Instructions - Clearing System | Our Delivery Instructions - Account | Client Delivery Instructions - SettlementAgent | Client Delivery Instructions - Account | Client Delivery Instructions - Agent | Client Delivery Instructions - Clearing | Consideration for settlement | Currency for Settlement | Disclaimer 1 | Disclaimer 2 | Disclaimer 3 | Definition of Business Day | Disclaimer 4 | Disclaimer 5 |
|-----------------------------------|-----------------------------------|--------------|---------|------------------|--------------|-----------------|-------------|------------|--------|-----------|----------------|------------|----------------------|-------------|------------------------|---------------------|-----------------|---------------------|------------------|-----------|-----------|------------|-------------------|---------------|----------------------|-------------------------------------------|-------------------------------------|----------------------------------------------|----------------------------------------|--------------------------------------|----------------------------------------|------------------------------|-------------------------|--------------|--------------|--------------|---------------------------|--------------|--------------|
| Email Notice of Execution message | This is a NEW Notice of Execution | SAMSUNG ASSET MANAGEMENT CO.,LTD. | SAMSUNG ASSET MANAGEMENT CO.,LTD. | globalop@samsung.com | AMEROps@bernsteinsg.com | 0000000216674524 | 20250826 09:30:00.913 | LMT | MLT *** | BYV2325 | 245 Park Avenue<br>New York, NY 10167 | (917) 344 8575 | S&P GLOBAL INC | Aug 26, 2025 | Charges | USD 137,031.45 | USD 41.11 | USD 0.00 | USD 0.00 | USD 0.00 | USD 0.00 | USD 136,990.34 | 1.000000000 | We will receive versus payment | Clearing system | DTC: 0286 | SettlementAgent | SBVQ | 29796 | 00997 | Consideration for settlement is 136,990.34 settling in USD (US Dollar) | USD (US Dollar) | Please do not reply to this mail. Should you have a query, please email Bernstein Institutional Services Trade Support on AMEROps@bernsteinsg.com or call (917) 344 8575, quoting the trade reference. | Please inform Bernstein Institutional Services Trade Support of any differences you might have with the present confirmation at the latest, within one Business Day of dispatch to you. | The lack of any objection on your part shall be deemed to be your agreement with the above information. | "Business Day" means a day which is not a Saturday or a Sunday and on which banks are open for business. | Please note this is not a regulatory confirmation and is provided for informational purposes only. | A more detailed confirmation compliant with SEC Rule 10b-10 will follow. |

---

## 3) 추출 데이터 검수 결과

- **문서 거래 건수(N) vs 출력 행 수**: 1건 → 출력 1행 ✅ 일치  
- **누락된 필수 필드 여부**: 모든 필수 필드(23개)가 채워짐. Clearing Broker, Settlement Location (PSET)은 원문에 명시되지 않아 `MISSING` 처리 ✅  
- **기본값 적용 검증**:  
  - Commission: 41.11 → 숫자만 추출 ✅  
  - Taxes: 0.00 → 원문 `USD 0.00` → `0.00` ✅  
  - Other Charges: 0.00 → 원문 모두 0.00 → `0.00` ✅  
  - Executed Qty: 249 → 소수점 제거 ✅  
- **날짜/금액/수량/B/S 포맷 검증**:  
  - Trade Date: `Aug 26, 2025` → `2025-08-26` ✅  
  - Settlement Date: `Aug 27, 2025` → `2025-08-27` ✅  
  - Deal Price: `550.3271` → 소수점 4자리 유지 ✅  
  - Gross Amount: `137031.45` → 쉼표 제거 ✅  
  - Net Settlement AMT: `136990.34` → 쉼표 제거 ✅  
  - B/S: `We have SOLD` → `S` ✅  
  - Currency: `USD` → 원문 그대로 ✅  
- **Fund Code 정규화 적용 검증**:  
  - Account Name: `BOK EQ ESG PASSIVE SAMSUNG(SAM_783011)` → Fund Code 추출: `SAM_783011` → `SAM_` 제거 → `783011` ✅  
  - Fund Name: Account Name 그대로 사용 → `BOK EQ ESG PASSIVE SAMSUNG(SAM_783011)` ✅  
  - Ticker: `SPGI` → 국가 코드 추가: `SPGI US` ✅  
- **기타 검증**:  
  - Broker Address: 줄바꿈 복원 → `245 Park Avenue<br>New York, NY 10167` → 기타 테이블에 원문 그대로 유지 ✅  
  - Commission: `USD 41.11` → 숫자만 추출 → `41.11` ✅  
  - 모든 디스클레임, 전화번호, 이메일, 코드 등 원문 그대로 유지 ✅  
- **최종 확인**:  
  - **모든 원문 데이터 보존** ✅  
  - **임의 수정/번역/요약 없음** ✅  
  - **정규화 및 포맷 적용 완료** ✅  
  - **Markdown 테이블 구조 오류 없음** ✅  
```


---
**토큰 사용량:**
- 입력 토큰: 6415
- 출력 토큰: 1793
- 총 토큰: 8208


In [27]:
print(data_markdown.content)

```markdown
## 1) 거래(Trade) 단위 정리 테이블

| Trade Date | Fund Name | Fund Code | Ticker | ISIN | Security Name | Settlement Date | B/S | Currency | Executed Qty | Deal Price | Gross Amount | Commission | Taxes | Other Charges | Net Settlement AMT | Executing Broker | Clearing Broker | Settlement Location (PSET) | Sec Account | Clearing Agent ID | Account |
|------------|-----------|-----------|--------|------|---------------|-----------------|-----|----------|--------------|------------|--------------|------------|-------|---------------|---------------------|------------------|-----------------|----------------------------|-------------|-------------------|---------|
| 2025-08-26 | BOK EQ ESG PASSIVE SAMSUNG(SAM_783011) | 783011 | SPGI US | US78409V1044 | S&P GLOBAL INC | 2025-08-27 | S | USD | 249 | 550.3271 | 137031.45 | 41.11 | 0.00 | 0.00 | 136990.34 | Bernstein Institutional Services, LLC | MISSING | MISSING | DTC: 0286 | 29796 | 7011253890 |

---

## 2) 기타 데이터 정리 테이블

| Email Notic